# Flight Delay Prediction

Flight disruptions cause substantial economic losses for airlines and significant inconvenience for travelers. In this project, we build a data mining pipeline that predicts domestic flight delays from historical flight records and weather forecast data.

To mitigate data sparsity across the U.S. aviation network, we restrict our scope to domestic flights between the **15 busiest U.S. airport hubs**.

The problem is framed in two ways:
- **Primary task — binary classification:** predict whether a flight arrives more than 15 minutes late.
- **Secondary task — regression:** predict the arrival delay in minutes.

To prevent data leakage, every predictive feature is limited to information available **2 hours before the scheduled departure (T − 2h)**. We use a chronological train/test split to mirror real-world forecasting, and compare a route-average baseline, symbolic models (Decision Trees, Random Forests) and non-symbolic models (SVM, XGBoost) using F1-score, ROC-AUC and MAE.

## Project structure

### 1: Data selection
- Choosing the subset of images that belong to our scope (top 15 U.S airport hubs).

### 2: Data preprocessing
- Missing Value Imputation
- Categorical Encoding
- Feature Normalization
- Class Imbalance Handling

### 3: Transformation
// TODO

### 4: Data Mining
// TODO

### 5: Interpretation and evaluation


## 1. Data selection

In [19]:
import pandas as pd

'''
    Since the dataframe has so many unwanted attributes, we need to choose which collumns or attributues we are interested in, the others will not go into
    the dataframe.
    Also, the last collumns have a great ammount of missing values, so indeed the best option is to drop them.
'''

COLS = [
    # time
    "FlightDate", "Month", "DayofMonth", "DayOfWeek",
    # flight / aircraft
    "Reporting_Airline", "Tail_Number", "Flight_Number_Reporting_Airline",
    # route
    "Origin", "Dest", "Distance",
    # schedule (known before departure)
    "CRSDepTime", "CRSArrTime", "CRSElapsedTime",
    # actual times (needed to build the incoming-aircraft feature, NOT as inputs)
    "DepTime", "ArrTime",
    # labels
    "ArrDelay", "ArrDelayMinutes", "ArrDel15",
    # filtering
    "Cancelled", "Diverted",
    # optional, analysis only
    "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay",
]

# FINDING TOP 15 BUSIEST AIRPORTS
df_all = pd.read_csv("data/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_1.csv",
                     usecols=["Origin", "Dest"])

counts = df_all["Origin"].value_counts()

HUBS = counts.head(15).index.tolist()


# Final dataframe we will use
df = pd.read_csv("data/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_1.csv", usecols=COLS,
                 dtype={"CRSDepTime": str, "CRSArrTime": str})

# Filtering
df = df[df["Origin"].isin(HUBS) & df["Dest"].isin(HUBS)]
df = df[(df["Cancelled"] == 0) & (df["Diverted"] == 0)]  # No cancelled or diverted flights (desviados)

# Showing 10 examples
pd.set_option("display.max_columns", None) 
pd.set_option("display.width", 200)

df.sample(10, random_state=42)

,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,CRSDepTime,DepTime,CRSArrTime,ArrTime,ArrDelay,ArrDelayMinutes,ArrDel15,Cancelled,Diverted,CRSElapsedTime,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
538822,1,7,3,2026-01-07,UA,N24514,2430,DEN,ORD,0750,739.0,1134,1104.0,-30.0,0.0,0.0,0.0,0.0,164.0,888.0,NaN,NaN,NaN,NaN,NaN
87472,1,29,4,2026-01-29,AS,N812AK,637,PHX,SEA,1100,1056.0,1319,1320.0,1.0,1.0,0.0,0.0,0.0,199.0,1107.0,NaN,NaN,NaN,NaN,NaN
329272,1,14,3,2026-01-14,WN,N456WN,528,LAS,PHX,1830,1824.0,2045,2025.0,-20.0,0.0,0.0,0.0,0.0,75.0,255.0,NaN,NaN,NaN,NaN,NaN
79926,1,5,1,2026-01-05,AS,N979AK,407,ORD,SEA,1533,1530.0,1816,1803.0,-13.0,0.0,0.0,0.0,0.0,283.0,1721.0,NaN,NaN,NaN,NaN,NaN
9516,1,18,7,2026-01-18,AA,N458AL,1925,MIA,DFW,1202,1154.0,1433,1437.0,4.0,4.0,0.0,0.0,0.0,211.0,1121.0,NaN,NaN,NaN,NaN,NaN
368678,1,22,4,2026-01-22,YX,N104HQ,4429,ATL,DCA,1404,1354.0,1559,1538.0,-21.0,0.0,0.0,0.0,0.0,115.0,547.0,NaN,NaN,NaN,NaN,NaN
125999,1,21,3,2026-01-21,DL,N509DT,450,MIA,LAX,0750,748.0,1045,1034.0,-11.0,0.0,0.0,0.0,0.0,355.0,2342.0,NaN,NaN,NaN,NaN,NaN
64433,1,23,5,2026-01-23,AA,N150UW,1722,CLT,MCO,1120,1116.0,1312,1257.0,-15.0,0.0,0.0,0.0,0.0,112.0,468.0,NaN,NaN,NaN,NaN,NaN
225503,1,30,5,2026-01-30,MQ,N225NN,3359,ORD,BOS,1152,1217.0,1517,1633.0,76.0,76.0,1.0,0.0,0.0,145.0,867.0,0.0,25.0,51.0,0.0,0.0
209955,1,2,5,2026-01-02,F9,N644FR,3142,ORD,CLT,0545,707.0,0852,950.0,58.0,58.0,1.0,0.0,0.0,127.0,599.0,58.0,0.0,0.0,0.0,0.0


In [20]:
print(f"After the data selection, we have {df.shape[0]} usable examples")

print(f"Top 15 busiest airports: {HUBS}")

After the data selection, we have 70627 usable examples
Top 15 busiest airports: ['ORD', 'ATL', 'DEN', 'DFW', 'PHX', 'LAX', 'CLT', 'MCO', 'LAS', 'SEA', 'DCA', 'SFO', 'LGA', 'MIA', 'BOS']
